## Imports

In [86]:
import os
import numpy as np
import torch
import torch.nn as nn
import pandas as pd
import cv2
import mediapipe as mp
import pandas as pd
import numpy as np
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
from IPython.display import HTML
import joblib

if torch.backends.mps.is_available():
    device = torch.device("mps")      # Mac GPU (Apple Silicon)
elif torch.cuda.is_available():
    device = torch.device("cuda")     # Nvidia GPU
else:
    device = torch.device("cpu")

## Normalize target 0-4 

In [87]:
import pandas as pd

# Load CSV
df = pd.read_csv("../../../scores.csv")

# -------------------------------------------------
# ORIGINAL RANGE
# -------------------------------------------------

old_min = 0.887892851954438
old_max = 1.335416

# -------------------------------------------------
# SCALE TO 0-4
# -------------------------------------------------

df["score_0_4"] = (
    (df["Var2"] - old_min) /
    (old_max - old_min)
) * 4

# -------------------------------------------------
# OPTIONAL: CLIP VALUES
# -------------------------------------------------

df["score_0_4"] = df["score_0_4"].clip(0, 4)

# -------------------------------------------------
# SAVE NEW CSV
# -------------------------------------------------

df = df.drop(columns=["Var2"])

df.to_csv("scores_scaled_0_4.csv", index=False)

print("Saved: scores_scaled_0_4.csv")

Saved: scores_scaled_0_4.csv


## Functions

In [ ]:
def create_fixed_c_sequence_from_running_column(
    input_folder,
    output_folder,
    score_csv_path,
    C=30
):
    os.makedirs(output_folder, exist_ok=True)

    score_df = pd.read_csv(score_csv_path)

    score_dict = {}

    for _, row in score_df.iterrows():

        video_id = str(row["Var1"]).split("_")[0]

        score_dict[video_id] = row["Var2"]

    joints = [
        "head",
        "left_shoulder", "left_elbow",
        "right_shoulder", "right_elbow",
        "left_hand", "right_hand",
        "left_hip", "right_hip",
        "left_knee", "right_knee",
        "left_foot", "right_foot"
    ]

    columns = []

    for frame_idx in range(C):
        for joint in joints:
            columns += [
                f"frame{frame_idx}_{joint}_x",
                f"frame{frame_idx}_{joint}_y",
                f"frame{frame_idx}_{joint}_z"
            ]

    columns.append("target")

    saved_count = 0

    for file_name in os.listdir(input_folder):

        if not file_name.endswith(".csv"):
            continue

        print(f"\nProcessing: {file_name}")

        video_id = file_name.split("_")[0]

        if video_id not in score_dict:
            print(f"Skipping {file_name}: no target found")
            continue

        csv_path = os.path.join(input_folder, file_name)

        df = pd.read_csv(csv_path)

        # -----------------------------------------
        # Trim using running_video column
        # -----------------------------------------

        running_df = df[df["running_video"] == 1]

        if len(running_df) == 0:
            print(f"Skipping {file_name}: no running frames")
            continue

        # -----------------------------------------
        # Remove non-feature columns
        # -----------------------------------------

        drop_cols = ["FrameNo", "running_video"]

        feature_df = running_df.drop(
            columns=[c for c in drop_cols if c in running_df.columns]
        )

        X = feature_df.values.astype(np.float32)

        # -----------------------------------------
        # Convert to fixed C frames
        # -----------------------------------------

        if len(X) < C:
            print(f"Skipping {file_name}: sequence too short")
            continue

        indices = np.linspace(0, len(X) - 1, C).astype(int)

        X_fixed = X[indices]

        # -----------------------------------------
        # Target
        # -----------------------------------------

        target = score_dict[video_id]

        # -----------------------------------------
        # Flatten + save
        # -----------------------------------------

        row = X_fixed.flatten().tolist()
        row.append(target)

        output_df = pd.DataFrame([row], columns=columns)

        output_path = os.path.join(
            output_folder,
            f"{video_id}_fixed_c.csv"
        )

        output_df.to_csv(output_path, index=False)

        saved_count += 1

        print(f"Trimmed frames: {len(X)}")
        print(f"Fixed shape: {X_fixed.shape}")
        print(f"Target: {target}")
        print(f"Saved: {output_path}")

    print("\nDone.")
    print(f"Saved {saved_count} videos.")

## Paths

In [ ]:
input_folder = "../../MainProject/Assignment11/data/mediapipe_not_cut_start_stop"
output_folder = "../../MainProject/data/mediapipe_score_fixed_c_score"

score_csv_path = "scores_scaled_0_4.csv"

## Run trought csv

In [ ]:
create_fixed_c_sequence_from_running_column(
    input_folder=input_folder,
    output_folder=output_folder,
    score_csv_path=score_csv_path,
    C=30
)


Processing: A75_mediapipe.csv
Predictions:
[0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
Start index: 1, Stop index: 99
Target score: 1.8089920904500496
Fixed shape: (30, 39)
Saved: ../../MainProject/data/mediapipe_score_fixed_c_score/A75_score_fixed_c30.csv

Processing: A121_mediapipe.csv
Predictions:
[1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1

In [94]:
df = pd.read_csv("../../MainProject/data/mediapipe_score_fixed_c_score/A2_score_fixed_c30.csv")

y = df["target"].values

X_flat = df.drop(columns=["target"]).values

max_frames = 30 
n_features = 39

X = X_flat.reshape(-1, max_frames, n_features)

print(X.shape)

print(X)
print(y)

(1, 30, 39)
[[[ 0.58468181  0.25094867 -0.26200053 ...  0.55631351  0.91630363
    0.12932895]
  [ 0.58510655  0.25118792 -0.20357044 ...  0.55616641  0.91753858
    0.11308981]
  [ 0.58631498  0.25068772 -0.27851644 ...  0.55582082  0.91533822
    0.11477437]
  ...
  [ 0.58275324  0.24927458 -0.26003033 ...  0.55542904  0.91670871
    0.14880638]
  [ 0.58290553  0.24649692 -0.26984042 ...  0.55524564  0.91248649
    0.13410892]
  [ 0.58247471  0.24568702 -0.25248089 ...  0.55410081  0.9115504
    0.12666677]]]
[1.03116733]
